In [15]:
import requests
from bs4 import BeautifulSoup
import re
import sqlite3
import json


### Database Stuffs

In [16]:
conn = sqlite3.connect('lectures_FS2026.db')
c = conn.cursor()

# Enable foreign key support
c.execute('PRAGMA foreign_keys = ON;')
c.execute('PRAGMA foreign_keys = OFF;')
c.execute('DROP TABLE IF EXISTS lecture_category_link')
c.execute('DROP TABLE IF EXISTS lecture_lecturer_link')
c.execute('DROP TABLE IF EXISTS lectures')
c.execute('DROP TABLE IF EXISTS lecturers')
c.execute('DROP TABLE IF EXISTS study_tracks')
c.execute('DROP TABLE IF EXISTS arrow_one_categories')
c.execute('DROP TABLE IF EXISTS arrow_two_categories')
c.execute('DROP TABLE IF EXISTS arrow_three_categories')
c.execute('PRAGMA foreign_keys = ON;')

c.execute('''
CREATE TABLE IF NOT EXISTS lectures (
  id INTEGER PRIMARY KEY,
  number TEXT,
  title TEXT,
  url TEXT,
  type TEXT,
  ects TEXT,
  hours TEXT,
  abstract TEXT,
  learning_objective TEXT,
  content TEXT,
  lecture_notes TEXT,
  literature TEXT,
  language TEXT,
  periodicity TEXT,
  competencies TEXT,
  performance_assessment TEXT
)
''')

c.execute('''
CREATE TABLE IF NOT EXISTS lecturers (
  id INTEGER PRIMARY KEY,
  name TEXT,
  url TEXT
)
''')

c.execute('''
CREATE TABLE IF NOT EXISTS lecture_lecturer_link (
  lecture_id INTEGER,
  lecturer_id INTEGER,
  FOREIGN KEY (lecture_id) REFERENCES lectures(id),
  FOREIGN KEY (lecturer_id) REFERENCES lecturers(id)  
)
''')

# Category tables
c.execute('''
CREATE TABLE IF NOT EXISTS study_tracks (
  id INTEGER PRIMARY KEY,
  name TEXT
)
''')

c.execute('''
CREATE TABLE IF NOT EXISTS arrow_one_categories (
  id INTEGER PRIMARY KEY,
  name TEXT
)
''')

c.execute('''
CREATE TABLE IF NOT EXISTS arrow_two_categories (
  id INTEGER PRIMARY KEY,
  name TEXT
)
''')

c.execute('''
CREATE TABLE IF NOT EXISTS arrow_three_categories (
  id INTEGER PRIMARY KEY,
  name TEXT
)
''')

c.execute('''
CREATE TABLE IF NOT EXISTS lecture_category_link (
  lecture_id INTEGER,
  study_tracks_id INTEGER,
  arrow_one_category_id INTEGER,
  arrow_two_category_id INTEGER,
  arrow_three_category_id INTEGER,
  FOREIGN KEY (lecture_id) REFERENCES lectures(id),
  FOREIGN KEY (study_tracks_id) REFERENCES study_tracks(id),
  FOREIGN KEY (arrow_one_category_id) REFERENCES arrow_one_categories(id),
  FOREIGN KEY (arrow_two_category_id) REFERENCES arrow_two_categories(id),
  FOREIGN KEY (arrow_three_category_id) REFERENCES arrow_three_categories(id)
)
''')


### Scrape the course listing webpage (where all courses are listed)

In [17]:
# url = f"https://www.vvz.ethz.ch/Vorlesungsverzeichnis/sucheLehrangebot.view?lang=en&lerneinheitscode=&deptId=&famname=&unterbereichAbschnittId=&seite=1&lerneinheitstitel=&rufname=&kpRange=0,999&lehrsprache=&bereichAbschnittId=&semkez=2025W&studiengangAbschnittId=&studiengangTyp=&ansicht=1&&katalogdaten=&wahlinfo="

# the URL contains the code for the semester: e.g. 2025W for the winter semester of 2025/2026, and 2026S for the summer semester of 2026

url = f"https://www.vvz.ethz.ch/Vorlesungsverzeichnis/sucheLehrangebot.view?lang=en&lerneinheitscode=&deptId=&famname=&unterbereichAbschnittId=&seite=1&lerneinheitstitel=&rufname=&kpRange=0,999&lehrsprache=&bereichAbschnittId=&semkez=2026S&studiengangAbschnittId=&studiengangTyp=&ansicht=1&&katalogdaten=&wahlinfo="
res = requests.get(url)
soup = BeautifulSoup(res.text, 'html.parser')
# print(soup.prettify())

In [18]:
tbody = soup.find_all('table')
tbody

[<table style="wAuto"><col style="width: 150px"/><col style="width: 330px"/><col style="width: 30px"/><col style="width: auto"/><col style="width: 40px"/><col style="width: auto"/><col style="width: 170px"/><tr><td class="td-level" colspan="7"><a href="sucheLehrangebot.view?abschnittId=120216&amp;semkez=2026S&amp;ansicht=1&amp;lang=en&amp;seite=1">Agricultural Sciences Bachelor</a> <a alt="Information" class="symbolInfo" href="https://www.usys.ethz.ch/en/studies/agricultural-sciences/bachelor.html" target="_blank"><img alt="Information" border="0" src="images/icon-info.png" title="Information"/></a> </td></tr><tr><td class="td-level" colspan="7"><img alt="" class="levelIndicator" src="images/arrow-level-indicator.png"/> <a href="sucheLehrangebot.view?abschnittId=120804&amp;semkez=2026S&amp;ansicht=1&amp;lang=en&amp;seite=1">2. Semester</a></td></tr><tr><td class="td-level" colspan="7"><img alt="" class="levelIndicator" src="images/arrow-level-indicator.png"/> <img alt="" class="levelIn

In [19]:
rows = soup.find_all('tr')
rows

[<tr><td class="td-level" colspan="7"><a href="sucheLehrangebot.view?abschnittId=120216&amp;semkez=2026S&amp;ansicht=1&amp;lang=en&amp;seite=1">Agricultural Sciences Bachelor</a> <a alt="Information" class="symbolInfo" href="https://www.usys.ethz.ch/en/studies/agricultural-sciences/bachelor.html" target="_blank"><img alt="Information" border="0" src="images/icon-info.png" title="Information"/></a> </td></tr>,
 <tr><td class="td-level" colspan="7"><img alt="" class="levelIndicator" src="images/arrow-level-indicator.png"/> <a href="sucheLehrangebot.view?abschnittId=120804&amp;semkez=2026S&amp;ansicht=1&amp;lang=en&amp;seite=1">2. Semester</a></td></tr>,
 <tr><td class="td-level" colspan="7"><img alt="" class="levelIndicator" src="images/arrow-level-indicator.png"/> <img alt="" class="levelIndicator" src="images/arrow-level-indicator.png"/> <a href="sucheLehrangebot.view?abschnittId=121464&amp;semkez=2026S&amp;ansicht=1&amp;lang=en&amp;seite=1">First Year Examinations</a></td></tr>,
 <tr>

In [20]:
def count_arrows(row_html):
    soup = BeautifulSoup(row_html, 'html.parser')
    arrow_imgs = soup.find_all('img', {'src': 'images/arrow-level-indicator.png'})
    return len(arrow_imgs)

### Scrape Lecture Detail webpage

In [21]:
def extract_info_table(soup):
    h1 = soup.find('h1')
    if not h1:
        return {}
    table = h1.find_next('table')
    if not table:
        return {}
    info = {}
    for row in table.find_all('tr'):
        cells = row.find_all('td')
        if len(cells) >= 2:
            label = cells[0].get_text(strip=True)
            value = cells[1].get_text(separator='\n', strip=True)
            info[label] = value
    return info


def extract_performance_assessment(soup):
    h3 = soup.find('h3', string=lambda x: x and 'Performance assessment' in x)
    if not h3:
        return None
    table = h3.find_next('table')
    if not table:
        return None
    rows = []
    for row in table.find_all('tr'):
        text = row.get_text(' ', strip=True)
        if text:
            rows.append(text)
    return ' | '.join(rows) if rows else None


def extract_competencies(soup):
    h3 = soup.find('h3', string=lambda x: x and 'Catalogue data' in x)
    if not h3:
        return None
    table = h3.find_next('table')
    rows = table.find_all('tr')
    for row in rows:
        cells = row.find_all('td')
        if len(cells) >= 2 and cells[0].get_text(strip=True) == 'Competencies':
            inner_table = cells[1].find('table')
            if not inner_table:
                return None
            competencies = []
            for subrow in inner_table.find_all('tr'):
                subcells = [td.get_text(strip=True) for td in subrow.find_all('td')]
                competencies.append(subcells)
            return json.dumps(competencies)
    return None


def extract_catalogue_data(soup):
    h3 = soup.find('h3', string=lambda x: x and 'Catalogue data' in x)
    if not h3:
        raise Exception("Could not find the 'Catalogue data' header.")
    table = h3.find_next('table')
    rows = table.find_all('tr')
    details_list = []
    for row in rows:
        cells = row.find_all('td')
        if len(cells) == 2:
            label = cells[0].get_text(strip=True)
            value = cells[1].get_text(separator='
', strip=True)
            if label != 'Competencies':  # handled separately
                details_list.append((label, value))
    return details_list


### Database addition functions

In [22]:
def lecture_data_to_database(number, title, title_url, type_, credits_, hours,
                              abstract, learning_objective, content, lecture_notes, literature,
                              language=None, periodicity=None, competencies=None, performance_assessment=None):
    c.execute('SELECT 1 FROM lectures WHERE number = ?', (number,))

    if not c.fetchone():
        c.execute('''
            INSERT INTO lectures (number, title, url, type, ects, hours,
                abstract, learning_objective, content, lecture_notes, literature,
                language, periodicity, competencies, performance_assessment)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (number, title, title_url, type_, credits_, hours,
              abstract, learning_objective, content, lecture_notes, literature,
              language, periodicity, competencies, performance_assessment))
        conn.commit()


def lecturer_to_database(lecturer_names, lecturer_urls):
    for name, url in zip(lecturer_names, lecturer_urls):
        c.execute('SELECT 1 FROM lecturers WHERE name = ? AND url = ?', (name, url))
        if not c.fetchone():
            c.execute('INSERT INTO lecturers (name, url) VALUES (?, ?)', (name, url))
            conn.commit()


def categories_to_database(study_track, arrow_level_one, arrow_level_two, arrow_level_three):
    if study_track:
        c.execute('SELECT 1 FROM study_tracks WHERE name = ?', (study_track,))
        if not c.fetchone():
            c.execute('INSERT INTO study_tracks (name) VALUES (?)', (study_track,))
            conn.commit()
    if arrow_level_one:
        c.execute('SELECT 1 FROM arrow_one_categories WHERE name = ?', (arrow_level_one,))
        if not c.fetchone():
            c.execute('INSERT INTO arrow_one_categories (name) VALUES (?)', (arrow_level_one,))
            conn.commit()
    if arrow_level_two:
        c.execute('SELECT 1 FROM arrow_two_categories WHERE name = ?', (arrow_level_two,))
        if not c.fetchone():
            c.execute('INSERT INTO arrow_two_categories (name) VALUES (?)', (arrow_level_two,))
            conn.commit()
    if arrow_level_three:
        c.execute('SELECT 1 FROM arrow_three_categories WHERE name = ?', (arrow_level_three,))
        if not c.fetchone():
            c.execute('INSERT INTO arrow_three_categories (name) VALUES (?)', (arrow_level_three,))
            conn.commit()


def link_lecture_lecturer(number, lecturer_names, lecturer_urls):
    c.execute('SELECT id FROM lectures WHERE number = ?', (number,))
    lecture_row = c.fetchone()
    if not lecture_row:
        print(f"Lecture with number {number} not found.")
        return
    lecture_id = lecture_row[0]

    for name, url in zip(lecturer_names, lecturer_urls):
        c.execute('SELECT id FROM lecturers WHERE name = ? AND url = ?', (name, url))
        lecturer_row = c.fetchone()
        if not lecturer_row:
            continue
        lecturer_id = lecturer_row[0]

        c.execute('''
            SELECT 1 FROM lecture_lecturer_link 
            WHERE lecture_id = ? AND lecturer_id = ?
        ''', (lecture_id, lecturer_id))

        if not c.fetchone():
            c.execute('''
                INSERT INTO lecture_lecturer_link (lecture_id, lecturer_id)
                VALUES (?, ?)
            ''', (lecture_id, lecturer_id))
            conn.commit()


def link_lecture_category(number, study_track, arrow_level_one, arrow_level_two, arrow_level_three):
    c.execute('SELECT id FROM lectures WHERE number = ?', (number,))
    lecture_row = c.fetchone()
    if not lecture_row:
        print(f"Lecture with number {number} not found.")
        return
    lecture_id = lecture_row[0]

    c.execute('SELECT id FROM study_tracks WHERE name = ?', (study_track,))
    study_track_row = c.fetchone()
    study_track_id = study_track_row[0] if study_track_row else None

    c.execute('SELECT id FROM arrow_one_categories WHERE name = ?', (arrow_level_one,))
    arrow_one_row = c.fetchone()
    arrow_one_id = arrow_one_row[0] if arrow_one_row else None

    c.execute('SELECT id FROM arrow_two_categories WHERE name = ?', (arrow_level_two,))
    arrow_two_row = c.fetchone()
    arrow_two_id = arrow_two_row[0] if arrow_two_row else None

    c.execute('SELECT id FROM arrow_three_categories WHERE name = ?', (arrow_level_three,))
    arrow_three_row = c.fetchone()
    arrow_three_id = arrow_three_row[0] if arrow_three_row else None

    c.execute('''
        SELECT 1 FROM lecture_category_link 
        WHERE lecture_id = ? AND 
              (study_tracks_id IS ? OR study_tracks_id IS NULL) AND
              (arrow_one_category_id IS ? OR arrow_one_category_id IS NULL) AND
              (arrow_two_category_id IS ? OR arrow_two_category_id IS NULL) AND
              (arrow_three_category_id IS ? OR arrow_three_category_id IS NULL)
    ''', (lecture_id, study_track_id, arrow_one_id, arrow_two_id, arrow_three_id))

    if not c.fetchone():
        c.execute('''
            INSERT INTO lecture_category_link (lecture_id, study_tracks_id, arrow_one_category_id, arrow_two_category_id, arrow_three_category_id)
            VALUES (?, ?, ?, ?, ?)
        ''', (lecture_id, study_track_id, arrow_one_id, arrow_two_id, arrow_three_id))
        conn.commit()


### main scraper function

In [23]:
def extract_lectures(soup):
    rows = soup.find_all('tr')

    study_track = None
    arrow_level_one = None
    arrow_level_two = None
    arrow_level_three = None

    for row in rows:
        cells = row.find_all('td')

        if len(cells) == 1:  # category header rows
            link = cells[0].find('a')
            text = link.get_text(strip=True) if link else ''
            arrow_count = count_arrows(str(row))

            if text == '':  # skip empty spacer rows
                continue

            if arrow_count == 0:
                study_track = text
                arrow_level_one = None
                arrow_level_two = None
                arrow_level_three = None
            elif arrow_count == 1:
                arrow_level_one = text
                arrow_level_two = None
                arrow_level_three = None
            elif arrow_count == 2:
                arrow_level_two = text
                arrow_level_three = None
            elif arrow_count == 3:
                arrow_level_three = text

        if len(cells) >= 6:
            number = cells[0].get_text(strip=True)

            title_link = cells[1].find('b').find('a') if cells[1].find('b') else None
            title = title_link.get_text(strip=True) if title_link else None

            url_temp = cells[1].find('a')['href'] if cells[1].find('a') else None
            if url_temp and 'lerneinheitId' in url_temp:
                match = re.search(r'lerneinheitId=(\d+)', url_temp)
                if match:
                    lerneinheit_id = match.group(1)
                    title_url = f'https://www.vvz.ethz.ch/Vorlesungsverzeichnis/lerneinheit.view?semkez=2026S&ansicht=ALLE&lerneinheitId={lerneinheit_id}&lang=en'
                else:
                    title_url = None
            else:
                title_url = None

            type_ = cells[2].get_text(strip=True)
            credits_ = cells[3].get_text(strip=True)
            hours = cells[4].get_text(strip=True)

            # Extract lecturer names from individual <a> tags (not by splitting text by comma)
            lecturer_links = cells[5].find_all('a')
            lecturer_names = [a.get_text(strip=True) for a in lecturer_links]
            lecturer_urls = ['https://www.vvz.ethz.ch' + a['href'] for a in lecturer_links if a.has_attr('href')]

            # Fetch detail page
            language = None
            periodicity = None
            competencies = None
            performance_assessment = None
            abstract = None
            learning_objective = None
            content = None
            lecture_notes = None
            literature = None

            if title_url:
                try:
                    res = requests.get(title_url, timeout=10)
                    detail_soup = BeautifulSoup(res.text, 'html.parser')

                    info = extract_info_table(detail_soup)
                    periodicity = info.get('Periodicity')
                    language = info.get('Language of instruction')

                    details = extract_catalogue_data(detail_soup)
                    details_dict = dict(details)
                    abstract = details_dict.get('Abstract') or None
                    learning_objective = details_dict.get('Learning objective') or None
                    content = details_dict.get('Content') or None
                    lecture_notes = details_dict.get('Lecture notes') or None
                    literature = details_dict.get('Literature') or None

                    competencies = extract_competencies(detail_soup)
                    performance_assessment = extract_performance_assessment(detail_soup)
                except Exception as e:
                    print(f'Warning: Could not fetch details for {number}: {e}')

            lecture_data_to_database(number, title, title_url, type_, credits_, hours,
                                    abstract, learning_objective, content, lecture_notes, literature,
                                    language, periodicity, competencies, performance_assessment)
            lecturer_to_database(lecturer_names, lecturer_urls)
            categories_to_database(study_track, arrow_level_one, arrow_level_two, arrow_level_three)

            link_lecture_lecturer(number, lecturer_names, lecturer_urls)
            link_lecture_category(number, study_track, arrow_level_one, arrow_level_two, arrow_level_three)

            print(f'Lecture Info: {number, title, type_, credits_, hours}')


In [24]:
# extract_lectures(soup)

### loop that fetches the next page, and calls the main scraper function

In [25]:
table_urls = []
table_urls.append(f"https://www.vvz.ethz.ch/Vorlesungsverzeichnis/sucheLehrangebot.view?lang=en&search=on&semkez=2026S&studiengangTyp=&deptId=&studiengangAbschnittId=&lerneinheitstitel=&lerneinheitscode=&famname=&rufname=&wahlinfo=&lehrsprache=&periodizitaet=&kpRange=0%2C999&katalogdaten=&_strukturAus=on&search=Search")
# old links:
# "https://www.vvz.ethz.ch/Vorlesungsverzeichnis/sucheLehrangebot.view?lang=en&lerneinheitscode=&deptId=&famname=&unterbereichAbschnittId=&seite=1&lerneinheitstitel=&rufname=&kpRange=0,999&lehrsprache=&bereichAbschnittId=&semkez=2025W&studiengangAbschnittId=&studiengangTyp=&ansicht=1&&katalogdaten=&wahlinfo="

for url in table_urls:
    res = requests.get(url)
    soup = BeautifulSoup(res.text, 'html.parser')

    # extract lecture information from the current page
    extract_lectures(soup)
    
    # get the next page url
    next_page_img = soup.find("img", class_="nextPage")
    next_page_link = next_page_img.find_parent("a")

    if not next_page_link:
        print("No more pages.")
    else:
        next_url = next_page_link['href']
        next_url = f"https://www.vvz.ethz.ch{next_url}"
        # print(next_url)
        table_urls.append(next_url)

Lecture Info: ('529-2002-02L', 'Chemistry II', 'O', '5\xa0credits', '2V\xa0+\xa02U')
Lecture Info: ('401-0252-00L', 'Mathematics II', 'O', '7\xa0credits', '5V\xa0+\xa02U')
Lecture Info: ('551-0002-00L', 'General Biology II', 'O', '4\xa0credits', '4G')
Lecture Info: ('751-0270-00L', 'Systematics and Ecology of Algae and Fungi', 'O', '2\xa0credits', '2G')
Lecture Info: ('751-0280-00L', 'Crops in the World Food System', 'O', '2\xa0credits', '2V')
Lecture Info: ('751-0282-00L', 'Animal Sciences in the World Food System', 'O', '2\xa0credits', '2V')
Lecture Info: ('751-0014-00L', 'Agricultural Economics in the World Food System', 'O', '2\xa0credits', '2V')
Lecture Info: ('851-0708-00L', 'Introduction to Law', 'O', '2\xa0credits', '2V')
Lecture Info: ('751-0304-00L', 'Excursions in the World Food System', 'O', '1\xa0credit', '2P')
Lecture Info: ('402-0062-00L', 'Physics I', 'O', '5\xa0credits', '3V\xa0+\xa01U')
Lecture Info: ('751-8001-00L', 'Agricultural Production Systems', 'O', '2\xa0credi

### Cells above this point contain the complete working scraper.\n
The exploration/debug cells below are retained for reference but not part of the pipeline.\n

### Close the Database connection

In [26]:
conn.commit()
conn.close()